# Model Training

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [2]:
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

In [3]:
spark = (
    SparkSession.builder
    .appName("BusServiceReliability")
    .master("local[*]")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/02 02:02:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/08/02 02:02:43 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/08/02 02:02:43 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/08/02 02:02:43 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/08/02 02:02:43 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.


In [4]:
df = spark.read.parquet("../outputs/cleaned_timetable_parquet")

In [5]:
df.show(5, truncate=False)

+-------------------------------------------------------------------------------------------------------+----------+----------+------------+-------------+----------+
|filename                                                                                               |journey_id|operator  |service_code|stop_sequence|route_size|
+-------------------------------------------------------------------------------------------------------+----------+----------+------------+-------------+----------+
|10A-None--SCMY-GM-2026-07-26-Gillmoss_July_2026_EV_ADDED_FINAL__SCMY_PC1033334_5_20260719-BODS_V1_1.xml|VJ2904    |Stagecoach|PC1033334:5 |55           |Long      |
|10A-None--SCMY-GM-2026-07-26-Gillmoss_July_2026_EV_ADDED_FINAL__SCMY_PC1033334_5_20260719-BODS_V1_1.xml|VJ2904    |Stagecoach|PC1033334:5 |56           |Long      |
|10A-None--SCMY-GM-2026-07-26-Gillmoss_July_2026_EV_ADDED_FINAL__SCMY_PC1033334_5_20260719-BODS_V1_1.xml|VJ2904    |Stagecoach|PC1033334:5 |57           |Long      |
|10A

## Preparing the Target Variable

In [6]:
from pyspark.sql.functions import when, col

df = df.withColumn(
    "target",
    when(col("route_size") == "Long", 1)
    .otherwise(0)
)

df.groupBy("target").count().show()

[Stage 2:>                                                        (0 + 12) / 12]

+------+------+
|target| count|
+------+------+
|     1| 37157|
|     0|234395|
+------+------+



In [7]:
df.select(
    "route_size",
    "target"
).show(10)

+----------+------+
|route_size|target|
+----------+------+
|      Long|     1|
|      Long|     1|
|      Long|     1|
|      Long|     1|
|      Long|     1|
|      Long|     1|
|      Long|     1|
|      Long|     1|
|      Long|     1|
|      Long|     1|
+----------+------+
only showing top 10 rows


## Feature Preparation

In [8]:
from pyspark.ml.feature import StringIndexer, VectorAssembler

In [9]:
indexer = StringIndexer(
    inputCol="service_code",
    outputCol="service_code_index"
)

df = indexer.fit(df).transform(df)

In [10]:
from pyspark.sql.functions import countDistinct, stddev

service_features = df.groupBy("service_code").agg(
    countDistinct("journey_id").alias("num_journeys"),
    stddev("stop_sequence").alias("stop_seq_std")
)
df = df.join(service_features, on="service_code", how="left")

df.select("service_code", "num_journeys", "stop_seq_std").show(5)

+------------+------------+-----------------+
|service_code|num_journeys|     stop_seq_std|
+------------+------------+-----------------+
| PC1033334:5|         275|19.04683747209288|
| PC1033334:5|         275|19.04683747209288|
| PC1033334:5|         275|19.04683747209288|
| PC1033334:5|         275|19.04683747209288|
| PC1033334:5|         275|19.04683747209288|
+------------+------------+-----------------+
only showing top 5 rows


In [11]:
# Re-confirm assembler is the 3-feature version
assembler = VectorAssembler(
    inputCols=["service_code_index", "num_journeys", "stop_seq_std"],
    outputCol="features"
)
dataset = assembler.transform(df)
train_data, test_data = dataset.randomSplit([0.8, 0.2], seed=42)

# Now retrain LR on this
lr = LogisticRegression(featuresCol="features", labelCol="target", maxIter=10)
lr_model = lr.fit(train_data)

print("New LR numFeatures:", lr_model.numFeatures)  # should print 3

lr_model.write().overwrite().save("logistic_regression_model")
print("Saved.")

26/08/02 02:03:14 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
                                                                                

New LR numFeatures: 3
Saved.


## Train-Test Split

In [12]:
train_data, test_data = dataset.randomSplit([0.8, 0.2], seed=42)

print("Training records:", train_data.count())
print("Testing records:", test_data.count())

Training records: 217633


[Stage 96:====>                                                   (1 + 11) / 12]

Testing records: 53919


In [13]:
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
import pandas as pd

## Train random forest

In [14]:
rf = RandomForestClassifier(featuresCol="features", labelCol="target", numTrees=20, seed=42, maxBins=128)
rf_model = rf.fit(train_data)

# Make predictions on test data
rf_predictions = rf_model.transform(test_data)

print("Random Forest trained successfully!")
rf_predictions.select("features", "target", "prediction", "probability").show(10, truncate=False)

Random Forest trained successfully!
+------------------------------+------+----------+----------------------------------------+
|features                      |target|prediction|probability                             |
+------------------------------+------+----------+----------------------------------------+
|[8.0,336.0,10.604775332999036]|0     |0.0       |[0.8634117969561487,0.13658820304385139]|
|[8.0,336.0,10.604775332999036]|0     |0.0       |[0.8634117969561487,0.13658820304385139]|
|[8.0,336.0,10.604775332999036]|0     |0.0       |[0.8634117969561487,0.13658820304385139]|
|[8.0,336.0,10.604775332999036]|0     |0.0       |[0.8634117969561487,0.13658820304385139]|
|[8.0,336.0,10.604775332999036]|0     |0.0       |[0.8634117969561487,0.13658820304385139]|
|[8.0,336.0,10.604775332999036]|0     |0.0       |[0.8634117969561487,0.13658820304385139]|
|[8.0,336.0,10.604775332999036]|0     |0.0       |[0.8634117969561487,0.13658820304385139]|
|[8.0,336.0,10.604775332999036]|0     |0.0  

## Train logistic regression

In [15]:
print("Training Logistic Regression...")
lr = LogisticRegression(featuresCol="features", labelCol="target", maxIter=10)
lr_model = lr.fit(train_data)
lr_predictions = lr_model.transform(test_data)

print("Saving models...")
lr_model.write().overwrite().save("logistic_regression_model")
rf_model.write().overwrite().save("random_forest_model")
print("Models saved successfully!")

Training Logistic Regression...
Saving models...
Models saved successfully!


In [17]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator_roc = BinaryClassificationEvaluator(labelCol="target", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

In [18]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

# Build a small hyperparameter grid for Logistic Regression
paramGrid = (
    ParamGridBuilder()
    .addGrid(lr.regParam, [0.01, 0.1, 0.5])
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0])
    .build()
)

cv = CrossValidator(
    estimator=lr,
    estimatorParamMaps=paramGrid,
    evaluator=BinaryClassificationEvaluator(labelCol="target", rawPredictionCol="rawPrediction", metricName="areaUnderROC"),
    numFolds=3,
    seed=42
)

cv_model = cv.fit(train_data)
print("Best regParam:", cv_model.bestModel._java_obj.getRegParam())
print("Best elasticNetParam:", cv_model.bestModel._java_obj.getElasticNetParam())

# Evaluate the tuned model
cv_predictions = cv_model.transform(test_data)
print("Tuned LR ROC-AUC:", evaluator_roc.evaluate(cv_predictions))

Best regParam: 0.01
Best elasticNetParam: 0.0


[Stage 2766:======================>                                (5 + 7) / 12]

Tuned LR ROC-AUC: 0.8484572073912896


## Train Decision Tree Classifier

In [20]:
dt = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="target",
    maxBins=128
)

## Model Efficiency — Interpretation

In [22]:
import time
import builtins
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

f1_evaluator = MulticlassClassificationEvaluator(labelCol="target", predictionCol="prediction", metricName="f1")

def time_and_score(estimator, model_name):
    start = time.time()
    fitted_model = estimator.fit(train_data)
    duration = time.time() - start
    preds = fitted_model.transform(test_data)
    f1 = f1_evaluator.evaluate(preds)
    efficiency = f1 / duration
    return {
        "Model": model_name,
        "Training Time (s)": builtins.round(duration, 2),
        "F1-score": builtins.round(f1, 4),
        "Model Efficiency (F1/sec)": builtins.round(efficiency, 4)
    }

efficiency_results = [
    time_and_score(lr, "Logistic Regression"),
    time_and_score(rf, "Random Forest"),
    time_and_score(dt, "Decision Tree")
]

efficiency_df = pd.DataFrame(efficiency_results)
print("\n✅ MODEL EFFICIENCY (F1-score per second of training) ✅\n")
display(efficiency_df)


✅ MODEL EFFICIENCY (F1-score per second of training) ✅



,Model,Training Time (s),F1-score,Model Efficiency (F1/sec)
0,Logistic Regression,3.45,0.8303,0.2403
1,Random Forest,5.81,0.7987,0.1374
2,Decision Tree,3.89,0.7987,0.2052


## Make Predictions

In [ ]:
predictions = model.transform(test_data)

In [ ]:
from pyspark.ml.classification import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="target",
    maxBins=128
)
dt_model = dt.fit(train_data)
dt_predictions = dt_model.transform(test_data)
dt_predictions.select("features", "target", "prediction", "probability").show(10, truncate=False)

## Feature Importance

In [ ]:
print("Feature Importances:")
print(model.featureImportances)

## Save Trained Model

In [ ]:
dt_model.write().overwrite().save("../models/decision_tree_model")
print("Decision Tree model saved successfully!")

# Summary

In this notebook, the cleaned dataset was prepared for machine learning by creating a target variable and assembling numerical features. A Decision Tree classifier was trained using the training dataset, predictions were generated for the testing dataset, feature importance was examined, and the trained model was saved for evaluation.